### ЗАДАЧА: Панель SLA-ребейтов по доставке по паттерну `MVC`

Команда logistics finance разбирает кейсы по SLA-ребейтам: если доставка опоздала,
клиенту или продавцу может быть положена компенсация, а к логистическому партнеру — применен штраф.
Нужно реализовать внутреннюю консольную панель по паттерну `MVC`.

Слои:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `promised_days` — обещанный срок доставки;
- `actual_days` — фактический срок доставки;
- `order_value` — стоимость заказа;
- `shipping_fee` — стоимость доставки;
- `penalty_rate` — ставка компенсации за каждый день просрочки;
- `delay_days` — число дней просрочки;
- `requested_rebate` — расчетная сумма ребейта;
- `approved_rebate` — согласованная сумма ребейта;
- `courier_penalty` — штраф, который будет предъявлен логистическому партнеру;
- `status` — статус кейса;
- `coordinator` — сотрудник, который ведет кейс;
- `customer_contacted` — связывались ли с клиентом;
- `decision` — финальное решение.

## Формулы

При создании кейса и после изменения одобренной суммы нужно считать:
- `delay_days = max(actual_days - promised_days, 0)`
- `requested_rebate = min(order_value * penalty_rate * delay_days, shipping_fee + order_value * 0.2)`
- `approved_rebate` при создании равно `0.0`
- `courier_penalty = approved_rebate * 0.7`
- все денежные значения округляются до 2 знаков.

## Статусы

- `new`
- `investigating`
- `customer_contacted`
- `ready_for_approval`
- `approved`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `coordinator` несуществующему кейсу;
- финальные кейсы (`approved`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `coordinator`;
- связаться с клиентом можно только из `investigating`;
- при контакте с клиентом поле `customer_contacted` должно стать `True`, а статус — `customer_contacted`;
- установить `approved_rebate` можно только из `investigating` или `customer_contacted`;
- `approved_rebate` не может быть меньше `0`;
- `approved_rebate` не может быть больше `requested_rebate`;
- после изменения `approved_rebate` нужно пересчитать `courier_penalty`;
- перевод в `ready_for_approval` возможен только из `investigating` или `customer_contacted`;
- перевод в `ready_for_approval` возможен только если `approved_rebate > 0`;
- завершить кейс как `approved` можно только из `ready_for_approval`;
- завершить кейс как `rejected` можно только из `ready_for_approval`, если `approved_rebate == 0`;
- `escalated` можно сделать только из `investigating`, `customer_contacted` или `ready_for_approval`;
- при финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать координатора;
- начинать расследование;
- отмечать контакт с клиентом;
- устанавливать `approved_rebate`;
- переводить кейс в `ready_for_approval`;
- завершать кейс как `approved`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | shipment_id | courier | promised_days | actual_days | delay_days | order_value | shipping_fee | requested_rebate | approved_rebate | courier_penalty | status | coordinator | customer_contacted | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_requested_rebate` — общая расчетная сумма ребейтов;
- `total_approved_rebate` — общая согласованная сумма;
- `total_courier_penalty` — общая сумма штрафов логистическому партнеру;
- `delayed_shipments` — количество кейсов, где `delay_days > 0`;
- `contacted_cases` — количество кейсов, где `customer_contacted == True`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести список кейсов и summary.

In [ ]:
initial_cases = [
    ("DR-100", "SHP-9901", "FastBox", 2, 5, 4200.0, 300.0, 0.03),
    ("DR-101", "SHP-9902", "QuickShip", 3, 3, 1800.0, 220.0, 0.02),
]

actions = [
    ("show",),
    ("investigate", "DR-100"),
    ("assign", "DR-100", "Olga"),
    ("investigate", "DR-100"),
    ("contact", "DR-100"),
    ("set_rebate", "DR-100", 180.0),
    ("ready", "DR-100"),
    ("approve", "DR-100", "rebate_sent_to_customer"),
    ("create", "DR-102", "SHP-9903", "CityRun", 1, 4, 2600.0, 180.0, 0.04),
    ("assign", "DR-102", "Max"),
    ("investigate", "DR-102"),
    ("set_rebate", "DR-102", 150.0),
    ("ready", "DR-102"),
    ("escalate", "DR-102", "courier_disputes_delay_window"),
    ("create", "DR-103", "SHP-9904", "ParcelWay", 2, 6, 5200.0, 450.0, 0.03),
    ("assign", "DR-103", "Ina"),
    ("investigate", "DR-103"),
    ("set_rebate", "DR-103", 0.0),
    ("ready", "DR-103"),
    ("reject", "DR-103", "no_customer_refund_required"),
    ("show",),
]

from dataclasses import dataclass, field
from typing import List, Dict, Optional

# --- Model ---

@dataclass
class Case:
    case_id: str
    shipment_id: str
    courier: str
    promised_days: int
    actual_days: int
    order_value: float
    shipping_fee: float
    penalty_rate: float

    delay_days: int = field(init=False, default=0)
    requested_rebate: float = field(init=False, default=0.0)
    approved_rebate: float = field(default=0.0)
    courier_penalty: float = field(init=False, default=0.0)
    status: str = field(default='new')
    coordinator: Optional[str] = None
    customer_contacted: bool = False
    decision: Optional[str] = None

    def __post_init__(self):
        self.calculate_delays_and_rebates()

    def calculate_delays_and_rebates(self):
        self.delay_days = max(self.actual_days - self.promised_days, 0)
        max_rebate = self.shipping_fee + self.order_value * 0.2
        self.requested_rebate = round(
            min(self.order_value * self.penalty_rate * self.delay_days, max_rebate), 2
        )
        # On creation, approved rebate is zero
        if not hasattr(self, 'approved_rebate'):
            self.approved_rebate = 0.0
        self.calculate_courier_penalty()

    def calculate_courier_penalty(self):
        self.courier_penalty = round(self.approved_rebate * 0.7, 2)


class Model:
    def __init__(self):
        self.cases: Dict[str, Case] = {}

    def create_case(self, case_id, shipment_id, courier, promised_days, actual_days,
                    order_value, shipping_fee, penalty_rate):
        if case_id in self.cases:
            raise ValueError(f"Case {case_id} already exists.")
        case = Case(
            case_id, shipment_id, courier, promised_days, actual_days,
            order_value, shipping_fee, penalty_rate
        )
        self.cases[case_id] = case

    def assign_coordinator(self, case_id, coordinator):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        case.coordinator = coordinator

    def start_investigation(self, case_id):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status not in ['new', 'investigating']:
            raise ValueError(f"Cannot start investigation for case {case_id} in status {case.status}")
        if not case.coordinator:
            raise ValueError(f"Case {case_id} has no coordinator assigned.")
        case.status = 'investigating'

    def contact_customer(self, case_id):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status != 'investigating':
            raise ValueError(f"Can only contact customer from 'investigating'; current status: {case.status}")
        case.customer_contacted = True
        case.status = 'customer_contacted'

    def set_approved_rebate(self, case_id, rebate_amount):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status not in ['investigating', 'customer_contacted']:
            raise ValueError(f"Cannot set rebate for case {case_id} in status {case.status}")
        if rebate_amount < 0:
            raise ValueError("Rebate cannot be negative.")
        if rebate_amount > case.requested_rebate:
            raise ValueError(f"Rebate {rebate_amount} exceeds requested {case.requested_rebate}.")
        case.approved_rebate = round(rebate_amount, 2)
        case.calculate_courier_penalty()

    def move_to_ready_for_approval(self, case_id):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status not in ['investigating', 'customer_contacted']:
            raise ValueError(f"Cannot move case {case_id} to 'ready_for_approval' from status {case.status}")
        if case.approved_rebate <= 0:
            raise ValueError(f"Approved rebate must be > 0 to move to 'ready_for_approval'")
        case.status = 'ready_for_approval'

    def finalize_approval(self, case_id):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status != 'ready_for_approval':
            raise ValueError(f"Case {case_id} not in 'ready_for_approval' state.")
        case.status = 'approved'
        case.decision = 'approved'

    def finalize_rejection(self, case_id):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status != 'ready_for_approval':
            raise ValueError(f"Case {case_id} not in 'ready_for_approval' state.")
        if case.approved_rebate != 0:
            raise ValueError("Rebate must be 0 to reject the case.")
        case.status = 'rejected'
        case.decision = 'rejected'

    def escalate_case(self, case_id, reason):
        case = self.cases.get(case_id)
        if not case:
            raise ValueError(f"Case {case_id} not found.")
        if case.status not in ['investigating', 'customer_contacted', 'ready_for_approval']:
            raise ValueError(f"Cannot escalate case {case_id} from status {case.status}")
        case.status = 'escalated'

    def get_cases(self):
        return list(self.cases.values())

    def get_summary(self):
        summary = {
            'status_counts': {},
            'total_requested_rebate': 0.0,
            'total_approved_rebate': 0.0,
            'total_courier_penalty': 0.0,
            'delayed_shipments': 0,
            'contacted_cases': 0
        }
        for case in self.cases.values():
            # Count status
            summary['status_counts'][case.status] = summary['status_counts'].get(case.status, 0) + 1
            # Sum total requested rebate
            summary['total_requested_rebate'] += case.requested_rebate
            # Sum total approved rebate
            summary['total_approved_rebate'] += case.approved_rebate
            # Sum courier penalties
            summary['total_courier_penalty'] += case.courier_penalty
            # Count delays
            if case.delay_days > 0:
                summary['delayed_shipments'] += 1
            # Count contacted
            if case.customer_contacted:
                summary['contacted_cases'] += 1

        # Format monetary values to 2 decimals
        summary['total_requested_rebate'] = round(summary['total_requested_rebate'], 2)
        summary['total_approved_rebate'] = round(summary['total_approved_rebate'], 2)
        summary['total_courier_penalty'] = round(summary['total_courier_penalty'], 2)
        return summary

# --- View ---

class View:
    def display_cases(self, cases: List[Case]):
        for c in cases:
            print(
                f"{c.case_id} | {c.shipment_id} | {c.courier} | {c.promised_days} | {c.actual_days} "
                f"| {c.delay_days} | {c.order_value} | {c.shipping_fee} | {c.requested_rebate} "
                f"| {c.approved_rebate} | {c.courier_penalty} | {c.status} | {c.coordinator or ''} "
                f"| {c.customer_contacted} | {c.decision or ''}"
            )

    def display_summary(self, summary: Dict):
        print("\n--- Summary ---")
        print("Status counts:")
        for status, count in summary['status_counts'].items():
            print(f"  {status}: {count}")
        print(f"Total requested rebate: {summary['total_requested_rebate']}")
        print(f"Total approved rebate: {summary['total_approved_rebate']}")
        print(f"Total courier penalty: {summary['total_courier_penalty']}")
        print(f"Delayed shipments: {summary['delayed_shipments']}")
        print(f"Contacted cases: {summary['contacted_cases']}")

    def show_message(self, message: str):
        print(f"[MESSAGE] {message}")

    def show_error(self, error: str):
        print(f"[ERROR] {error}")

# --- Controller ---

class Controller:
    def __init__(self, model: Model, view: View):
        self.model = model
        self.view = view

    def create(self, case_id, shipment_id, courier, promised_days, actual_days,
               order_value, shipping_fee, penalty_rate):
        try:
            self.model.create_case(case_id, shipment_id, courier, promised_days,
                                   actual_days, order_value, shipping_fee, penalty_rate)
            self.view.show_message(f"Case {case_id} created.")
        except Exception as e:
            self.view.show_error(str(e))

    def assign(self, case_id, coordinator):
        try:
            self.model.assign_coordinator(case_id, coordinator)
            self.view.show_message(f"Coordinator {coordinator} assigned to {case_id}.")
        except Exception as e:
            self.view.show_error(str(e))

    def investigate(self, case_id):
        try:
            self.model.start_investigation(case_id)
            self.view.show_message(f"Investigation started for {case_id}.")
        except Exception as e:
            self.view.show_error(str(e))

    def contact(self, case_id):
        try:
            self.model.contact_customer(case_id)
            self.view.show_message(f"Customer contacted for {case_id}.")
        except Exception as e:
            self.view.show_error(str(e))

    def set_rebate(self, case_id, rebate):
        try:
            self.model.set_approved_rebate(case_id, rebate)
            self.view.show_message(f"Rebate {rebate} set for {case_id}.")
        except Exception as e:
            self.view.show_error(str(e))

    def ready_for_approval(self, case_id):
        try:
            self.model.move_to_ready_for_approval(case_id)
            self.view.show_message(f"Case {case_id} moved to 'ready_for_approval'.")
        except Exception as e:
            self.view.show_error(str(e))

    def approve(self, case_id, decision):
        try:
            self.model.finalize_approval(case_id)
            self.model.cases[case_id].decision = decision
            self.view.show_message(f"Case {case_id} approved.")
        except Exception as e:
            self.view.show_error(str(e))

    def reject(self, case_id, decision):
        try:
            self.model.finalize_rejection(case_id)
            self.model.cases[case_id].decision = decision
            self.view.show_message(f"Case {case_id} escalated for reason: {decision}.")
        except Exception as e:
            self.view.show_error(str(e))
    def escalate(self, case_id, reason):
        try:
            self.model.escalate_case(case_id, reason)
            self.view.show_message(f"Case {case_id} escalated for reason: {reason}.")
        except Exception as e:
            self.view.show_error(str(e))

    def show_cases(self):
        cases = self.model.get_cases()
        self.view.display_cases(cases)

    def show_summary(self):
        summary = self.model.get_summary()
        self.view.display_summary(summary)   


model = Model()
view = View()
controller = Controller(model, view)

controller.create('C001', 'SHP001', 'DHL', 3, 5, 1000.0, 50.0, 0.1)
controller.create('C002', 'SHP002', 'FedEx', 2, 2, 500.0, 30.0, 0.2)
controller.create('C003', 'SHP003', 'UPS', 4, 6, 2000.0, 100.0, 0.15)

# Назначение координаторов
controller.assign('C001', 'Alex')
controller.assign('C002', 'Maria')
controller.assign('C003', 'John')

# Начинаем расследование и контакты клиентов
controller.investigate('C001')
controller.contact('C001')
controller.set_rebate('C001', 20.0)
controller.ready_for_approval('C001')
controller.approve('C001', 'approved')

controller.investigate('C002')
controller.contact('C002')
controller.set_rebate('C002', 10.0)
controller.ready_for_approval('C002')
controller.reject('C002', 'rejected')

controller.investigate('C003')
controller.contact('C003')
controller.set_rebate('C003', 50.0)
controller.ready_for_approval('C003')
controller.escalate('C003', 'Disputed rebate amount')

# Выводим текущий список кейсов и сводку
controller.show_cases()
controller.show_summary()
    
    

[MESSAGE] Case C001 created.
[MESSAGE] Case C002 created.
[MESSAGE] Case C003 created.
[MESSAGE] Coordinator Alex assigned to C001.
[MESSAGE] Coordinator Maria assigned to C002.
[MESSAGE] Coordinator John assigned to C003.
[MESSAGE] Investigation started for C001.
[MESSAGE] Customer contacted for C001.
[MESSAGE] Rebate 20.0 set for C001.
[MESSAGE] Case C001 moved to 'ready_for_approval'.
[MESSAGE] Case C001 approved.
[MESSAGE] Investigation started for C002.
[MESSAGE] Customer contacted for C002.
[ERROR] Rebate 10.0 exceeds requested 0.0.
[ERROR] Approved rebate must be > 0 to move to 'ready_for_approval'
[ERROR] Case C002 not in 'ready_for_approval' state.
[MESSAGE] Investigation started for C003.
[MESSAGE] Customer contacted for C003.
[MESSAGE] Rebate 50.0 set for C003.
[MESSAGE] Case C003 moved to 'ready_for_approval'.
[MESSAGE] Case C003 escalated for reason: Disputed rebate amount.
C001 | SHP001 | DHL | 3 | 5 | 2 | 1000.0 | 50.0 | 200.0 | 20.0 | 14.0 | approved | Alex | True | app